# Summary_Day16_online.ipynb  
## CNN 스크래치 구현 · SimpleCNN · SVHN · Hook · Residual Block · ResNet18 감각

이번 16강은 **CNN을 사전학습 모델 없이 직접 설계하고 학습하는 흐름**을 정리하는 강의다.

13~15강에서는 pretrained 모델을 많이 사용했다.  
이번 강의에서는 다시 기본으로 돌아가서 `Conv2d`, `ReLU`, `MaxPool2d`, `Flatten`, `Linear`를 직접 쌓아 CNN을 만든다.

강의 핵심 흐름은 다음이다.

```text
MLP가 이미지에 약한 이유
→ Local Connectivity
→ Parameter Sharing
→ Spatial Hierarchy
→ Convolution / Filter / Kernel
→ Stride / Padding
→ Feature Map
→ Activation / Pooling
→ Feature Extractor와 Classifier 분리
→ CIFAR-10 SimpleCNN 직접 구현
→ SVHN 데이터셋으로 CNN 구현
→ Sequential 방식과 class 방식 비교
→ He 초기화
→ forward hook으로 layer 출력 shape 추적
→ 하이퍼파라미터 실험
→ Residual Block과 ResNet18 기본 감각
```

이 파일은 **인터넷 가능 버전**이다.  
CIFAR-10과 SVHN 데이터셋 다운로드가 가능하다는 전제로 작성했다.

> 필기 포인트:  
> pretrained 모델을 잘 쓰려면 결국 CNN 내부 구조를 알아야 한다.  
> 16강은 “남이 만든 모델을 쓰기 전, 내가 CNN 블록을 직접 설계할 수 있는가”를 확인하는 수업이다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. MLP가 이미지에서 가지는 한계를 이해한다.
2. CNN의 핵심 개념인 지역 연결, 파라미터 공유, 계층적 특징 학습을 정리한다.
3. CIFAR-10에서 SimpleCNN을 직접 만든다.
4. SVHN에서 `nn.Sequential` 방식과 class 기반 CNN 방식을 비교한다.
5. feature extractor와 classifier를 분리해서 모델을 설계한다.
6. layer별 출력 shape과 파라미터 수를 계산한다.
7. forward hook으로 중간 layer 출력 shape을 추적한다.
8. AdamW, weight decay, Dropout, He 초기화의 목적을 정리한다.
9. Residual Block이 왜 필요한지와 ResNet18 기본 감각을 잡는다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
```

- `torch`: Tensor 계산과 GPU 사용에 필요하다.
- `nn`: `Conv2d`, `Linear`, `MaxPool2d`, `Dropout` 같은 layer를 만든다.
- `optim`: Adam, AdamW 같은 optimizer를 만든다.
- `F`: `F.relu()`처럼 함수형 활성화 함수를 쓸 때 사용한다.
- `datasets`: CIFAR-10, SVHN 같은 표준 데이터셋을 불러온다.
- `transforms`: 이미지 전처리와 증강을 순서대로 묶는다.
- `DataLoader`: Dataset을 mini-batch 단위로 공급한다.
- `Subset`: 전체 데이터 중 일부만 골라 빠르게 실습한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)

## 3. device 설정과 시드 고정

### 함수 사용법

```python
torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

- GPU가 있으면 `cuda`를 사용한다.
- GPU가 없으면 `cpu`를 사용한다.
- 모델과 Tensor는 같은 device에 있어야 한다.

```python
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
```

- 난수를 고정해서 실험 결과가 너무 흔들리지 않게 한다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

print("device:", device)

## 4. CNN이 필요한 이유 정리

강의 초반의 핵심은 MLP의 한계다.

MLP는 이미지를 펼쳐서 하나의 긴 벡터로 만든다.

```text
32 × 32 × 3 = 3072개 feature
```

이 방식의 문제는 다음이다.

- 인접 픽셀 간의 관계를 잃는다.
- 위치가 조금만 바뀌어도 다른 패턴처럼 볼 수 있다.
- 모든 픽셀을 모든 뉴런과 연결하면 파라미터 수가 커진다.
- 파라미터가 많아지면 연산량과 과적합 위험이 커진다.

CNN은 이 문제를 다음 세 가지 아이디어로 줄인다.

```text
Local Connectivity    → 가까운 픽셀만 먼저 본다
Parameter Sharing     → 같은 필터를 이미지 전체에 재사용한다
Spatial Hierarchy     → 낮은 층은 선/모서리, 높은 층은 복잡한 형태를 본다
```

> 강사님 흐름:  
> CNN은 이미지의 지역적 패턴을 보고, 같은 필터를 반복해서 쓰며, 층이 깊어질수록 더 복잡한 특징을 학습한다.

## 5. Convolution 용어 정리

| 용어 | 뜻 |
|---|---|
| Filter / Kernel | 이미지 위를 움직이며 특징을 뽑는 작은 행렬 |
| Stride | 필터가 이동하는 간격 |
| Padding | 가장자리에 0 등을 추가해 크기 감소를 조절하는 방법 |
| Feature Map | 필터를 적용한 결과 |
| Channel | RGB 색상 축 또는 feature 축 |
| Pooling | 공간 크기를 줄이고 중요한 값만 남기는 연산 |

중요한 shape 감각은 다음이다.

```text
입력: [batch, channel, height, width]
Conv2d 출력: [batch, out_channels, new_height, new_width]
```

`out_channels`는 필터 개수이고, 출력 feature map 개수를 결정한다.

In [ ]:
# Conv2d shape 감각 확인
dummy = torch.randn(4, 3, 32, 32)

conv = nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3,
    stride=1,
    padding=1
)

pool = nn.MaxPool2d(kernel_size=2, stride=2)

conv_out = conv(dummy)
pool_out = pool(conv_out)

print("입력 shape:", dummy.shape)
print("Conv2d 출력 shape:", conv_out.shape)
print("MaxPool2d 출력 shape:", pool_out.shape)
print("Conv2d weight shape:", conv.weight.shape)

코드 해석:

```text
[4, 3, 32, 32]
→ Conv2d(3→32, kernel=3, padding=1)
→ [4, 32, 32, 32]
→ MaxPool2d(2)
→ [4, 32, 16, 16]
```

`padding=1` 덕분에 Conv 후 높이와 너비가 유지된다.  
Pooling을 지나면서 가로세로가 절반으로 줄어든다.

## 6. CIFAR-10 데이터 준비

첫 번째 실습은 CIFAR-10을 이용한 SimpleCNN이다.

CIFAR-10은 다음 특징을 가진다.

```text
이미지 크기: 32×32
채널 수: RGB 3채널
class 수: 10개
```

### 함수 사용법

```python
datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
```

- `root`: 데이터 저장 위치다.
- `train=True`: 학습 데이터다.
- `train=False`: 테스트 데이터다.
- `download=True`: 없으면 다운로드한다.
- `transform`: 이미지에 적용할 전처리다.

In [ ]:
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cifar_train_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=cifar_transform
)

cifar_test_full = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=cifar_transform
)

# 빠른 실습용 subset이다. 전체 학습을 원하면 이 부분을 제거한다.
cifar_train = Subset(cifar_train_full, list(range(5000)))
cifar_test = Subset(cifar_test_full, list(range(1000)))

cifar_train_loader = DataLoader(cifar_train, batch_size=64, shuffle=True)
cifar_test_loader = DataLoader(cifar_test, batch_size=64, shuffle=False)

cifar_classes = ("비행기", "자동차", "새", "고양이", "사슴", "개", "개구리", "말", "배", "트럭")

print("CIFAR train:", len(cifar_train))
print("CIFAR test:", len(cifar_test))

## 7. CIFAR-10 SimpleCNN 모델 만들기

강의 첫 번째 모델은 feature extractor와 classifier를 분리한다.

```text
features:
Conv2d(3→32) → ReLU → MaxPool
Conv2d(32→64) → ReLU → MaxPool

classifier:
Flatten
Linear(64×8×8 → 256)
ReLU
Linear(256 → 10)
```

### 함수 사용법

```python
nn.Sequential(...)
```

- 여러 layer를 순서대로 묶는다.
- 입력이 첫 layer부터 마지막 layer까지 순서대로 통과한다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

simple_model = SimpleCNN().to(device)

print(simple_model)

dummy = torch.randn(1, 3, 32, 32).to(device)

with torch.no_grad():
    feature_out = simple_model.features(dummy)
    output = simple_model(dummy)

print("features 출력:", feature_out.shape)
print("model 출력:", output.shape)

## 8. SimpleCNN 학습/평가 함수

학습 함수의 기본 순서는 항상 같다.

```text
model.train()
→ optimizer.zero_grad()
→ outputs = model(inputs)
→ loss = criterion(outputs, labels)
→ loss.backward()
→ optimizer.step()
```

평가 함수는 gradient 계산이 필요 없으므로 `torch.no_grad()`를 사용한다.

In [ ]:
def train_simple(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        pred = torch.max(outputs, 1)[1]

        running_loss += loss.item() * labels.size(0)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def evaluate_simple(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            pred = torch.max(outputs, 1)[1]

            running_loss += loss.item() * labels.size(0)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total

## 9. SimpleCNN 짧은 학습 실행

강의 원본은 5 epoch를 실행한다.  
여기서는 빠른 실습을 위해 기본 2 epoch로 둔다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_model.parameters(), lr=0.001)

num_epochs = 2

cifar_history = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_simple(
        simple_model,
        cifar_train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc = evaluate_simple(
        simple_model,
        cifar_test_loader,
        criterion,
        device
    )

    cifar_history.append([epoch, train_loss, train_acc, test_loss, test_acc])

    print(
        f"epoch {epoch} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.2f}% | "
        f"test_loss={test_loss:.4f} | test_acc={test_acc:.2f}%"
    )

cifar_history = np.array(cifar_history)

In [ ]:
plt.plot(cifar_history[:, 0], cifar_history[:, 1], label="train loss")
plt.plot(cifar_history[:, 0], cifar_history[:, 3], label="test loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("CIFAR-10 SimpleCNN Loss")
plt.legend()
plt.show()

plt.plot(cifar_history[:, 0], cifar_history[:, 2], label="train acc")
plt.plot(cifar_history[:, 0], cifar_history[:, 4], label="test acc")
plt.xlabel("epoch")
plt.ylabel("accuracy (%)")
plt.title("CIFAR-10 SimpleCNN Accuracy")
plt.legend()
plt.show()

그래프 해석:

- loss가 내려가면 모델이 정답 class에 가까워지고 있다는 뜻이다.
- accuracy가 올라가면 분류를 더 많이 맞히고 있다는 뜻이다.
- train accuracy만 높고 test accuracy가 낮으면 과적합을 의심한다.

## 10. CIFAR-10 예측 시각화

모델이 실제 이미지에서 어떻게 예측하는지 확인한다.

In [ ]:
def show_cifar_predictions(model, loader, classes, device, n_show=8):
    model.eval()

    images, labels = next(iter(loader))
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.max(outputs, 1)[1]

    images = images.cpu()
    labels = labels.cpu()
    preds = preds.cpu()

    plt.figure(figsize=(12, 6))

    for i in range(n_show):
        plt.subplot(2, 4, i + 1)

        img = images[i].permute(1, 2, 0) * 0.5 + 0.5
        img = torch.clamp(img, 0, 1)

        color = "blue" if labels[i].item() == preds[i].item() else "red"

        plt.imshow(img)
        plt.title(
            f"정답: {classes[labels[i].item()]}\n예측: {classes[preds[i].item()]}",
            color=color,
            fontsize=9
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_cifar_predictions(simple_model, cifar_test_loader, cifar_classes, device, n_show=8)

## 11. SVHN 데이터셋 준비

두 번째 실습은 SVHN이다.

SVHN은 Street View House Numbers의 약자다.  
구글 Street View에서 수집한 집 번호 이미지 데이터셋이다.

```text
이미지 크기: 32×32
채널 수: RGB 3채널
class 수: 10개, 숫자 0~9
```

강사님은 SVHN이 설정값이 조금 많은 Google 스타일 데이터셋이라고 설명했다.  
초기 설정은 많지만, 한 번 세팅하면 이후 학습 흐름은 CIFAR-10과 거의 같다.

In [ ]:
svhn_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

svhn_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

svhn_train_full = datasets.SVHN(
    root="./data",
    split="train",
    download=True,
    transform=svhn_train_transform
)

svhn_test_full = datasets.SVHN(
    root="./data",
    split="test",
    download=True,
    transform=svhn_test_transform
)

svhn_train = Subset(svhn_train_full, list(range(6000)))
svhn_test = Subset(svhn_test_full, list(range(1000)))

svhn_train_loader = DataLoader(svhn_train, batch_size=128, shuffle=True, num_workers=0)
svhn_test_loader = DataLoader(svhn_test, batch_size=128, shuffle=False, num_workers=0)

svhn_classes = [str(i) for i in range(10)]

print("SVHN train:", len(svhn_train))
print("SVHN test:", len(svhn_test))

## 12. SVHN 샘플 이미지 확인

데이터가 제대로 들어왔는지 먼저 확인한다.

### 함수 사용법

```python
next(iter(loader))
```

- DataLoader에서 첫 mini-batch를 하나 꺼낸다.

In [ ]:
def denormalize_half(tensor):
    return tensor * 0.5 + 0.5


def show_svhn_samples(loader, class_names, n_show=16):
    images, labels = next(iter(loader))

    plt.figure(figsize=(8, 8))

    for i in range(n_show):
        plt.subplot(4, 4, i + 1)

        img = denormalize_half(images[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)

        plt.imshow(img)
        plt.title(f"Label: {class_names[labels[i].item()]}", fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_svhn_samples(svhn_train_loader, svhn_classes, n_show=16)

## 13. SVHN CNN 모델 직접 구현

강의의 SVHN CNN은 3개의 convolution block과 classifier로 구성된다.

```text
Conv1: 3 → 32, 32×32 유지
Pool1: 32×32 → 16×16

Conv2: 32 → 64, 16×16 유지
Pool2: 16×16 → 8×8

Conv3: 64 → 128, 8×8 유지
Pool3: 8×8 → 4×4

Flatten: 128×4×4 = 2048
FC1: 2048 → 512
Dropout: p=0.5
FC2: 512 → 10
```

### 함수 사용법

```python
F.relu(x)
```

- `torch.nn.functional`의 ReLU 함수다.
- layer 객체를 따로 만들지 않고 함수처럼 적용한다.

In [ ]:
class SVHN_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(in_features=128 * 4 * 4, out_features=512)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(in_features=512, out_features=num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = self.pool3(x)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)

        return x

svhn_model = SVHN_CNN(num_classes=10).to(device)

print(svhn_model)

dummy = torch.randn(1, 3, 32, 32).to(device)

with torch.no_grad():
    out = svhn_model(dummy)

print("dummy output:", out.shape)

## 14. 파라미터 수 계산

### 함수 사용법

```python
p.numel()
```

- Tensor 안의 원소 개수를 센다.
- 모델 parameter 수 계산에 사용한다.

```python
p.requires_grad
```

- 학습 가능한 parameter인지 확인한다.

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total

total_params = count_parameters(svhn_model)

print(f"모델 총 파라미터 수: {total_params:,}개")

파라미터 계산 감각:

```text
Conv2d 파라미터 수 = (kernel_h × kernel_w × in_channels + bias 1) × out_channels
Linear 파라미터 수 = (in_features + bias 1) × out_features
```

예를 들어 첫 Conv는 다음이다.

```text
(3 × 3 × 3 + 1) × 32 = 896
```

이런 계산을 할 줄 알면 모델 크기와 과적합 위험을 더 잘 판단할 수 있다.

## 15. forward hook으로 layer 출력 shape 추적

강사님은 hook을 “중간 layer에 리본이나 CCTV를 달아두는 것”처럼 설명했다.  
모델 안쪽에서 Tensor가 어떤 shape으로 흐르는지 확인하는 도구다.

### 함수 사용법

```python
handle = module.register_forward_hook(hook_fn)
handle.remove()
```

- `register_forward_hook`: layer가 forward될 때 실행할 함수를 등록한다.
- `handle.remove()`: hook을 제거한다.
- hook을 제거하지 않으면 계속 남아서 메모리나 출력이 꼬일 수 있다.

In [ ]:
def trace_layer_shapes(model, input_shape=(1, 3, 32, 32), device="cpu"):
    layer_outputs = OrderedDict()
    handles = []

    def hook_fn(module, input, output):
        layer_name = module.__class__.__name__

        count = sum(1 for key in layer_outputs.keys() if layer_name in key)

        if count > 0:
            layer_name = f"{layer_name}_{count + 1}"

        if isinstance(output, torch.Tensor):
            layer_outputs[layer_name] = tuple(output.shape)

    for name, module in model.named_modules():
        if len(list(module.children())) == 0 and module != model:
            handle = module.register_forward_hook(hook_fn)
            handles.append(handle)

    dummy_input = torch.randn(*input_shape).to(device)

    model.eval()

    with torch.no_grad():
        _ = model(dummy_input)

    for handle in handles:
        handle.remove()

    return layer_outputs

shape_trace = trace_layer_shapes(svhn_model, input_shape=(1, 3, 32, 32), device=device)

for layer_name, shape in shape_trace.items():
    print(f"{layer_name:<20} {shape}")

출력 해석:

- Conv2d 뒤에는 channel 수가 바뀐다.
- MaxPool2d 뒤에는 가로세로 크기가 줄어든다.
- Linear 뒤에는 class 점수 방향으로 바뀐다.
- hook은 복잡한 모델에서 shape 오류를 찾을 때 매우 유용하다.

## 16. SVHN 학습/평가 함수

SVHN 학습에서는 tqdm 진행률 표시와 함께 loss, accuracy를 계산한다.

### 핵심 패턴

```text
model.train()
→ batch 반복
→ forward
→ loss
→ backward
→ optimizer.step()
→ running_loss, correct 누적
```

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc="학습", leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        predicted = outputs.argmax(dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def evaluate(model, test_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="평가", leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            predicted = outputs.argmax(dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy

## 17. SVHN 짧은 학습 실행

원본 강의는 10 epoch를 기준으로 한다.  
여기서는 실행 시간을 줄이기 위해 2 epoch로 둔다.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    svhn_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

num_epochs = 2

train_losses = []
train_accs = []
test_losses = []
test_accs = []

best_acc = 0.0
best_path = "/mnt/data/best_svhn_model_day16.pth"

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        svhn_model,
        svhn_train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc = evaluate(
        svhn_model,
        svhn_test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    print(f"epoch {epoch + 1}")
    print(f"학습 - loss: {train_loss:.4f}, accuracy: {train_acc:.2f}%")
    print(f"평가 - loss: {test_loss:.4f}, accuracy: {test_acc:.2f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(svhn_model.state_dict(), best_path)
        print(f"best model 저장: {best_acc:.2f}%")

In [ ]:
epochs_range = range(1, num_epochs + 1)

plt.plot(epochs_range, train_losses, label="train loss", marker="o")
plt.plot(epochs_range, test_losses, label="test loss", marker="s")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("SVHN Loss Curve")
plt.legend()
plt.show()

plt.plot(epochs_range, train_accs, label="train acc", marker="o")
plt.plot(epochs_range, test_accs, label="test acc", marker="s")
plt.xlabel("epoch")
plt.ylabel("accuracy (%)")
plt.title("SVHN Accuracy Curve")
plt.legend()
plt.show()

print("최고 테스트 정확도:", best_acc)

## 18. SVHN 예측 결과 시각화

저장된 best model을 불러와 테스트 이미지를 예측한다.

In [ ]:
def show_svhn_predictions(model, loader, class_names, device, model_path=None, n_show=16):
    if model_path is not None and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))

    model.eval()

    images, labels = next(iter(loader))
    images_device = images.to(device)

    with torch.no_grad():
        outputs = model(images_device)
        predicted = outputs.argmax(dim=1).cpu()

    plt.figure(figsize=(10, 10))

    for i in range(n_show):
        plt.subplot(4, 4, i + 1)

        img = denormalize_half(images[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)

        true_label = class_names[labels[i].item()]
        pred_label = class_names[predicted[i].item()]

        color = "blue" if true_label == pred_label else "red"

        plt.imshow(img)
        plt.title(f"실제: {true_label}\n예측: {pred_label}", color=color, fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_svhn_predictions(svhn_model, svhn_test_loader, svhn_classes, device, model_path=best_path, n_show=16)

## 19. Confusion Matrix와 클래스별 정확도

혼동 행렬은 어떤 숫자를 어떤 숫자로 헷갈렸는지 보여준다.

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="예측 수집", leave=False):
            inputs = inputs.to(device)

            outputs = model(inputs)
            predicted = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(predicted)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = collect_predictions(svhn_model, svhn_test_loader, device)

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("SVHN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(10), svhn_classes)
plt.yticks(range(10), svhn_classes)

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

plt.colorbar()
plt.show()

print(classification_report(y_true, y_pred, target_names=svhn_classes, zero_division=0))

## 20. Sequential 방식과 class 방식 비교

강의에서는 CNN을 두 방식으로 구현한다.

### 1. `nn.Sequential` 방식

```python
model = nn.Sequential(feature_extractor, classifier)
```

- 코드가 짧고 빠르게 만들기 좋다.
- 순서대로만 흐르는 모델에 적합하다.

### 2. class 방식

```python
class ClassCNN(nn.Module):
    def forward(self, x):
        ...
```

- forward 흐름을 직접 제어할 수 있다.
- hook, skip connection, 여러 입력/출력, 복잡한 구조에 적합하다.

> 시험 포인트:  
> 단순 모델은 Sequential이 편하고, 복잡한 모델은 class 방식이 좋다.

In [ ]:
seq_feature_extractor = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

seq_classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(128 * 4 * 4, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

seq_model = nn.Sequential(seq_feature_extractor, seq_classifier).to(device)

dummy = torch.randn(1, 3, 32, 32).to(device)

with torch.no_grad():
    out = seq_model(dummy)

print("Sequential 출력 shape:", out.shape)
print("Sequential 파라미터 수:", count_parameters(seq_model))

## 21. He 초기화 적용한 class CNN

He 초기화는 ReLU 계열 활성화 함수와 같이 자주 쓰인다.

### 함수 사용법

```python
nn.init.kaiming_normal_(m.weight)
nn.init.zeros_(m.bias)
```

- `kaiming_normal_`: ReLU에 적합한 weight 초기화다.
- `zeros_`: bias를 0으로 초기화한다.

In [ ]:
class ClassCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight)

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class_model = ClassCNN().to(device)

print("ClassCNN 파라미터 수:", count_parameters(class_model))

## 22. 하이퍼파라미터 실험 감각

강의에서 강조한 점은 하이퍼파라미터에는 정답이 하나로 고정되어 있지 않다는 것이다.

대표 CNN 하이퍼파라미터는 다음이다.

| 하이퍼파라미터 | 의미 |
|---|---|
| `ch1`, `ch2` | Conv layer의 출력 채널 수 |
| `kernel_size` | 필터 크기 |
| `stride` | 필터 이동 간격 |
| `padding` | 가장자리 보정 |
| `learning_rate` | 학습률 |
| `batch_size` | batch 크기 |
| `weight_decay` | L2 정규화 강도 |

채널 수가 많으면 표현력이 좋아질 수 있지만 파라미터 수와 과적합 위험도 늘어난다.  
커널 크기가 커지면 더 넓은 영역을 보지만 계산량도 증가한다.

In [ ]:
class SmallExp(nn.Module):
    def __init__(self, ch1=16, ch2=32, k=3, stride=1):
        super().__init__()

        pad = k // 2

        self.net = nn.Sequential(
            nn.Conv2d(3, ch1, k, stride=stride, padding=pad),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(ch1, ch2, k, stride=1, padding=pad),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(ch2 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

settings = [
    {"ch1": 16, "ch2": 32, "k": 3, "stride": 1},
    {"ch1": 32, "ch2": 64, "k": 3, "stride": 1},
    {"ch1": 32, "ch2": 64, "k": 5, "stride": 1}
]

for setting in settings:
    model_tmp = SmallExp(**setting).to(device)
    print(setting, "params:", count_parameters(model_tmp))

## 23. Residual Block 기본 구현

PDF 제목에 나온 Residual Block과 ResNet18의 핵심은 **skip connection**이다.

일반 CNN은 layer를 깊게 쌓을수록 gradient가 약해지는 문제가 생길 수 있다.  
ResNet은 다음처럼 입력 `x`를 block 출력에 더한다.

```text
out = F(x) + x
```

이 구조 덕분에 깊은 네트워크에서도 정보와 gradient가 더 잘 흐른다.

### shape이 같을 때

```text
입력 x와 F(x)의 shape이 같으면 그대로 더한다.
```

### shape이 다를 때

```text
1×1 Conv로 shortcut의 channel/size를 맞춘다.
```

In [ ]:
class BasicResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = F.relu(out)

        return out

block_same = BasicResidualBlock(32, 32, stride=1).to(device)
block_down = BasicResidualBlock(32, 64, stride=2).to(device)

x = torch.randn(4, 32, 32, 32).to(device)

with torch.no_grad():
    y_same = block_same(x)
    y_down = block_down(x)

print("입력:", x.shape)
print("same block 출력:", y_same.shape)
print("downsample block 출력:", y_down.shape)

출력 해석:

- `stride=1`, channel 동일이면 입력과 출력 shape이 같다.
- `stride=2`, channel 변경이면 공간 크기는 줄고 channel은 늘어난다.
- shortcut도 같은 shape이 되도록 1×1 Conv를 사용한다.

이것이 ResNet 계열 모델의 기본 block 감각이다.

## 24. TinyResNet 구조 감각

ResNet18 전체를 직접 구현하려면 block을 여러 개 쌓아야 한다.  
여기서는 작게 줄인 TinyResNet으로 구조 감각만 확인한다.

In [ ]:
class TinyResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.layer1 = BasicResidualBlock(32, 32, stride=1)
        self.layer2 = BasicResidualBlock(32, 64, stride=2)
        self.layer3 = BasicResidualBlock(64, 128, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

tiny_resnet = TinyResNet(num_classes=10).to(device)

dummy = torch.randn(2, 3, 32, 32).to(device)

with torch.no_grad():
    out = tiny_resnet(dummy)

print(tiny_resnet)
print("TinyResNet 출력:", out.shape)
print("TinyResNet 파라미터 수:", count_parameters(tiny_resnet))

## 25. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `CNN` | Convolutional Neural Network | 이미지 분류 모델 |
| `MLP` | Multi-Layer Perceptron | 이미지를 펼쳐 쓰는 기본 신경망 |
| `Conv2d` | 2D 합성곱 layer | `nn.Conv2d(in_ch, out_ch, k)` |
| `kernel` | 필터 크기 | 3×3, 5×5 등 |
| `stride` | 필터 이동 간격 | 1 또는 2 등 |
| `padding` | 가장자리 보정 | 크기 유지에 사용 |
| `feature map` | 필터 적용 결과 | 출력 channel과 연결 |
| `MaxPool2d` | 최대 pooling | 공간 크기 축소 |
| `Flatten` | 1차원 펼치기 | classifier 입력 |
| `Linear` | 완전연결층 | class 점수 계산 |
| `F.relu` | 함수형 ReLU | `F.relu(x)` |
| `Dropout` | 일부 뉴런 비활성화 | 과적합 완화 |
| `AdamW` | Adam + weight decay | `optim.AdamW(...)` |
| `weight_decay` | L2 정규화 계열 | 과적합 완화 |
| `SVHN` | Street View House Numbers | 숫자 이미지 데이터셋 |
| `hook` | 중간 layer 출력 추적 | `register_forward_hook` |
| `OrderedDict` | 순서 유지 딕셔너리 | layer 출력 저장 |
| `kaiming_normal_` | He 초기화 | ReLU 계열에 적합 |
| `Residual Block` | skip connection block | `F(x) + x` |
| `shortcut` | 건너뛰는 연결 | shape 맞춤 필요 |
| `ResNet` | Residual Network | 깊은 CNN 구조 |

## 26. 시험용 요약

```text
16강 핵심 = CNN을 직접 설계하고 layer별 shape, 파라미터, 학습 흐름을 이해하는 것
```

꼭 기억할 것:

- MLP는 이미지를 1차원으로 펼쳐 공간 정보를 잃는다.
- CNN은 지역 연결, 파라미터 공유, 계층적 특징 학습을 활용한다.
- `Conv2d`의 `out_channels`는 feature map 개수다.
- `padding=1`은 3×3 Conv에서 공간 크기를 유지할 때 자주 쓴다.
- `MaxPool2d(2)`는 가로세로를 절반으로 줄인다.
- feature extractor는 이미지 특징을 뽑는 부분이다.
- classifier는 뽑힌 특징을 class로 바꾸는 부분이다.
- CIFAR-10은 32×32 RGB 10 class 데이터셋이다.
- SVHN은 거리뷰 집 번호 숫자 이미지 데이터셋이다.
- `nn.Sequential`은 단순 순차 모델에 편하다.
- class 방식은 forward 흐름을 직접 제어할 수 있다.
- `x.view(x.size(0), -1)`은 batch를 유지하고 나머지를 펼친다.
- Conv 파라미터 수는 `(k×k×in_channels + 1) × out_channels`다.
- Linear 파라미터 수는 `(in_features + 1) × out_features`다.
- hook은 중간 layer 출력 shape을 추적하는 도구다.
- He 초기화는 ReLU 계열 모델에서 자주 쓴다.
- Residual Block은 `F(x) + x` 구조다.
- shape이 다르면 shortcut에 1×1 Conv를 사용한다.
- ResNet은 깊은 CNN에서 gradient 흐름을 돕는 구조다.